Det er modtaget! Det er en meget spændende specifikation, du har lavet der. "ApplicateCli" minder om arkitekturen bag systemer som Shopify eller moderne Headless CMS'er, hvor man definerer en "Schema" (Resource Spec), og systemet dynamisk genererer alt andet.

Det passer perfekt, at vi først får styr på HTTP, DI og LINQ i dybden, for du får brug for dem til at bygge din Engine, QueryExecutor og API lag. Især LINQ bliver vitalt, når du skal bygge din dynamiske QueryExecutor.

Her er de "Deep Dive" noter, du bad om. Du kan kopiere indholdet direkte ind i tre nye Polyglot Notebook filer.

Fil 1: HttpClient_BestPractices.ipynb
Dette er guiden til at tale med omverdenen uden at vælte din server.

Markdown Celle:
HTTP Client & Networking Standards
At hente data fra nettet virker simpelt, men det er en af de hyppigste kilder til fejl i produktionsmiljøer.

De 3 store fælder:

Socket Exhaustion: Hvis du bruger using (var client = new HttpClient()), lukker du forbindelsen, men Windows/Linux holder porten "reserveret" i 4 minutter (TIME_WAIT). Gør du det hurtigt nok, løber maskinen tør for porte.

DNS Rot: Hvis du laver en static HttpClient, opdager den aldrig, hvis serveren skifter IP-adresse.

Manglende Resiliency: Hvad hvis nettet blinker? Vi skal bruge "Polly" (Retry policies), men her fokuserer vi på klienten.

Løsningen: IHttpClientFactory Den styrer livscyklussen for "Handlers" (selve forbindelsen) separat fra "Clients" (det objekt du bruger).

C# Celle:
C#

using Microsoft.Extensions.DependencyInjection;
using Microsoft.Extensions.Hosting;
using Microsoft.Extensions.Http;

// --- 1. THE BAD WAY (Do not use in production loops) ---
// Dette virker til små scripts, men dræber servere under load.
async Task TheBadWay()
{
    using var client = new HttpClient(); // Opretter ny socket hver gang
    var html = await client.GetStringAsync("https://example.com");
    Console.WriteLine("Bad way finished.");
}

// --- 2. THE RIGHT WAY (Dependency Injection Setup) ---
var builder = Host.CreateApplicationBuilder();

// A. Basic Factory
builder.Services.AddHttpClient();

// B. Named Client (God til specifikke konfigurationer)
builder.Services.AddHttpClient("GithubClient", client => 
{
    client.BaseAddress = new Uri("https://api.github.com/");
    client.DefaultRequestHeaders.Add("User-Agent", "TaskFlowApp");
});

// C. Typed Client (BEST PRACTICE) -> Dette er vejen frem for din 'ResourceService'!
// Vi binder en C# klasse til en specifik HttpClient konfiguration.
public class WeatherService
{
    private readonly HttpClient _client;
    public WeatherService(HttpClient client) => _client = client;

    public async Task<string> GetForecast() => await _client.GetStringAsync("/weather");
}

// Registrering af Typed Client
builder.Services.AddHttpClient<WeatherService>(client => 
{
    client.BaseAddress = new Uri("https://api.weather.com/");
});

using var host = builder.Build();

// --- 3. USAGE ---
using (var scope = host.Services.CreateScope())
{
    // Hent Factory
    var factory = scope.ServiceProvider.GetRequiredService<IHttpClientFactory>();
    
    // Opret klient (Factory genbruger sockets under motorhjelmen!)
    var client = factory.CreateClient("GithubClient"); 
    Console.WriteLine($"Client Base Address: {client.BaseAddress}");
    
    // Eller hent den Typede Service
    var weather = scope.ServiceProvider.GetRequiredService<WeatherService>();
    // await weather.GetForecast(); // (Ville fejle her da URL er fake, men princippet holder)
}
Fil 2: Dependency_Injection_DeepDive.ipynb
Her går vi under motorhjelmen på containeren. Forståelse af "Scopes" og "Captive Dependencies" er forskellen på en junior og en senior .NET udvikler.

Markdown Celle:
Dependency Injection: Deep Dive
DI Containeren er rygraden i .NET. Den styrer hukommelse og levetid for dine objekter.

De 3 Livstider (Lifetimes)
Transient (AddTransient): "Letvægter".

Containeren laver en NY instans hver eneste gang, nogen spørger.

Analogi: En paptallerken. Du får en ny hver gang du skal spise.

Scoped (AddScoped): "Sessionen".

Containeren laver én instans pr. Scope (typisk pr. HTTP Request i web, eller pr. kommando i CLI).

Alle services i samme request deler den samme instans.

Analogi: En indkøbsvogn. Du bruger den samme gennem hele turen i supermarkedet, men næste kunde får en ny.

Singleton (AddSingleton): "Den udødelige".

Oprettes første gang den bruges, og lever indtil programmet lukker.

Analogi: Selve supermarkedet. Alle kunder deler den samme bygning.

Fælden: "Captive Dependency" ⚠️
Dette er den farligste fejl i DI. Hvis du putter en Scoped service ind i en Singleton, bliver den Scoped service fanget ("captive"). Den dør aldrig, selvom den burde dø efter requestet. Det skaber memory leaks og bugs.

In [ ]:
using Microsoft.Extensions.DependencyInjection;
using Microsoft.Extensions.Hosting;
var builder = Host.CreateApplicationBuilder();

// Setup
builder.Services.AddTransient<TransientService>();
builder.Services.AddScoped<ScopedService>();
builder.Services.AddSingleton<SingletonService>();
using var host = builder.Build();


// --- EKSPERIMENT 1: Scope vs Transient ---
Console.WriteLine("--- Scope 1 Starter ---");
using (var scope1 = host.Services.CreateScope())
{
    var t1 = scope1.ServiceProvider.GetRequiredService<TransientService>();
    var t2 = scope1.ServiceProvider.GetRequiredService<TransientService>();
    Console.WriteLine($"Transient (Ny hver gang): {t1.Id} vs {t2.Id}"); // Forskellige ID'er

    var s1 = scope1.ServiceProvider.GetRequiredService<ScopedService>();
    var s2 = scope1.ServiceProvider.GetRequiredService<ScopedService>();
    Console.WriteLine($"Scoped (Samme i scope):  {s1.Id} vs {s2.Id}"); // Ens ID'er!
    
    var single = scope1.ServiceProvider.GetRequiredService<SingletonService>();
    Console.WriteLine($"Singleton: {single.Id}");
}

Console.WriteLine("\n--- Scope 2 Starter (Nyt request) ---");
using (var scope2 = host.Services.CreateScope())
{
    var s3 = scope2.ServiceProvider.GetRequiredService<ScopedService>();
    Console.WriteLine($"Scoped (Nyt scope = Ny instans): {s3.Id}"); // Nyt ID i forhold til s1/s2
    
    var single2 = scope2.ServiceProvider.GetRequiredService<SingletonService>();
    Console.WriteLine($"Singleton (Altid den samme): {single2.Id}"); // Samme ID som i Scope 1
}
// Definitioner til test
public interface IService { string Id { get; } }

public class TransientService : IService { public string Id { get; } = Guid.NewGuid().ToString().Substring(0,4); }
public class ScopedService : IService    { public string Id { get; } = Guid.NewGuid().ToString().Substring(0,4); }
public class SingletonService : IService { public string Id { get; } = Guid.NewGuid().ToString().Substring(0,4); }


// --- EKSPERIMENT 2: Captive Dependency (DON'T DO THIS) ---
// Hvis SingletonService tog ScopedService i sin constructor, ville ScopedService
// leve evigt, fordi Singleton aldrig dør og slipper grebet.
// .NET advarer dig ofte om dette hvis "ValidateScopes" er slået til (default i Dev).